## Multimodal-RAG (PDF : Text + Images)

In [ ]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
import base64
import io

In [ ]:
# --- CLIP setup ---
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).eval()
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model.to(device)

# --- CLIP Embeddings ---
def embed_image(image_data):
    if isinstance(image_data, str):  # path
        image = Image.open(image_data)
    else:
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().cpu().numpy()


def embed_text(text):
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    ).to(device)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().cpu().numpy()

In [ ]:
# --- PDF Loading ---
pdf_path = "add_path_to_pdf"
doc = fitz.open(pdf_path)

all_docs = []
all_embeddings = []
image_data_store = {}
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

for i, page in enumerate(doc):
    text = page.get_text()
    if text.strip():
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            image_id = f"page_{i}_img_{img_index}"
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64

            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()

In [ ]:
# --- Initialize LLaVA ---
from transformers import AutoProcessor
from llava import LlavaForCausalLM  # Use the correct LLaVA class

llm_model_name = "llava-hf/llava-v1.6-vicuna-7b-hf"
processor = AutoProcessor.from_pretrained(llm_model_name)
llava_model = LlavaForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# --- LLaVA Inference ---
def llm_inference(query, images=[]):
    inputs = processor(images=images, text=query, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = llava_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs.get("pixel_values", None),
            max_new_tokens=200
        )
    return processor.decode(outputs[0], skip_special_tokens=True)

# --- Multimodal Retrieval ---
vector_store = None

def retrieve_multimodal(query, k=5):
    query_embedding = embed_text(query)
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    return results

def multimodal_pdf_rag_pipeline(query):
    context_docs = retrieve_multimodal(query, k=5)
    text_docs = [doc for doc in context_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in context_docs if doc.metadata.get("type") == "image"]

    text_context = "\n\n".join([
        f"[Page {doc.metadata['page']}]: {doc.page_content}"
        for doc in text_docs
    ])

    images_to_pass = []
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            image_bytes = base64.b64decode(image_data_store[image_id])
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            images_to_pass.append(pil_image)

    response_text = llm_inference(
        query=f"Question: {query}\nContext:\n{text_context}",
        images=images_to_pass
    )

    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")

    return response_text

In [ ]:
# -- Example --
if __name__ == "__main__":
    queries = [
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)